# Benchmark do `GIVPOptimizer.jl` — Funções Clássicas com Solução Conhecida

Notebook de exemplos do **GRASP-ILS-VND-PR** usando o pacote Julia **GIVPOptimizer.jl**.
Aplicado a problemas clássicos de otimização global cujo ótimo é conhecido analiticamente.

**Pré-requisito**: Julia instalado e `IJulia` disponível.
Se precisar instalar o pacote local execute:
```julia
import Pkg; Pkg.develop(path="../julia")
```

## 1. Setup e Importações

In [1]:
import Pkg

# Usa o pacote local quando possível, sem tentar "develop" do próprio projeto ativo
julia_pkg_path = abspath(joinpath(@__DIR__, "..", "julia"))
project_toml = abspath(joinpath(julia_pkg_path, "Project.toml"))
active_project = isnothing(Base.active_project()) ? "" : abspath(Base.active_project())

if active_project == project_toml
    println("Projeto Julia ativo já é o GIVPOptimizer local; pulando Pkg.develop.")
elseif isdir(julia_pkg_path)
    Pkg.develop(path=julia_pkg_path)
else
    # Fallback: tenta usar a versão registrada no General Registry
    Pkg.add("GIVPOptimizer")
end

using GIVPOptimizer
using Printf
using Statistics

println("Projeto ativo: ", Base.active_project())
println("GIVPOptimizer versão: ", GIVPOptimizer.__version__)
println("Setup concluído.")

Projeto Julia ativo já é o GIVPOptimizer local; pulando Pkg.develop.
Projeto ativo: d:\Projetos\Projeto_SOG2\grasp_ils_vnd_pr\julia\Project.toml
GIVPOptimizer versão: 0.5.2
Setup concluído.


## 2. Helper Genérico para Benchmark

Executa N rodadas independentes do otimizador e retorna métricas consolidadas.

In [2]:
struct BenchResult
    problem::String
    best_fun::Float64
    mean_fun::Float64
    std_fun::Float64
    err_abs::Float64
    err_rel_pct::Float64
    mean_time_s::Float64
    best_x::Vector{Float64}
    nfev::Int
end

function default_config()
    GIVPConfig(
        max_iterations=60,
        alpha=0.15,
        vnd_iterations=30,
        ils_iterations=8,
        perturbation_strength=4,
        num_candidates_per_step=15,
        elite_size=5,
        path_relink_frequency=10,
        early_stop_threshold=40,
        adaptive_alpha=true,
    )
end

function bench(
    name::String,
    fn::Function,
    bounds::Vector;
    optimum::Float64,
    n_runs::Int=3,
    direction::GIVPOptimizer.Direction=GIVPOptimizer.minimize,
    cfg=nothing,
    rel_tol::Float64=1.0,
)::BenchResult
    config = cfg === nothing ? default_config() : cfg
    values = Float64[]
    times = Float64[]
    best_x = Float64[]
    best_nfev = 0

    for _ in 1:n_runs
        t0 = time()
        res = givp(fn, bounds; direction=direction, config=config, seed=42)
        elapsed = time() - t0
        push!(values, res.fun)
        push!(times, elapsed)
        if isempty(best_x) || res.fun < first(values)
            best_x = res.x
            best_nfev = res.nfev
        end
    end

    best_val = direction == GIVPOptimizer.maximize ? maximum(values) : minimum(values)
    err_abs = abs(best_val - optimum)
    denom = max(abs(optimum), rel_tol)
    err_rel = 100.0 * err_abs / denom

    BenchResult(
        name, best_val, mean(values), std(values),
        err_abs, err_rel, mean(times),
        isempty(best_x) ? Float64[] : best_x,
        best_nfev,
    )
end

function print_result(r::BenchResult)
    @printf("── %s ──\n", r.problem)
    @printf("   melhor obtido  : %.6e\n", r.best_fun)
    @printf("   média ± dp     : %.6e ± %.3e\n", r.mean_fun, r.std_fun)
    @printf("   erro absoluto  : %.3e\n", r.err_abs)
    @printf("   erro rel (%%)   : %.4f%%\n", r.err_rel_pct)
    @printf("   tempo médio(s) : %.2f\n", r.mean_time_s)
    @printf("   nfev           : %d\n", r.nfev)
end

ALL_RESULTS = BenchResult[]
DEFAULT_RUNS = 3
println("Helper definido.")

Helper definido.


## 3. Problema 1: Sphere

$$f(x) = \sum_{i=1}^{n} x_i^2, \quad x \in [-5.12, 5.12]^{10}, \quad f^* = 0$$

In [3]:
sphere(x::Vector{Float64}) = sum(xi^2 for xi in x)

bounds_sphere = [(-5.12, 5.12) for _ in 1:10]
r_sphere = bench("Sphere", sphere, bounds_sphere; optimum=0.0, n_runs=DEFAULT_RUNS)
push!(ALL_RESULTS, r_sphere)
print_result(r_sphere)

── Sphere ──
   melhor obtido  : 2.009044e-03
   média ± dp     : 2.009044e-03 ± 0.000e+00
   erro absoluto  : 2.009e-03
   erro rel (%)   : 0.2009%
   tempo médio(s) : 2.42
   nfev           : 82747


## 4. Problema 2: Rosenbrock

$$f(x) = \sum_{i=1}^{n-1}\left[100(x_{i+1}-x_i^2)^2 + (1-x_i)^2\right],
\quad x^* = (1,\dots,1),\ f^* = 0$$

In [4]:
function rosenbrock(x::Vector{Float64})::Float64
    sum(100.0 * (x[i+1] - x[i]^2)^2 + (1.0 - x[i])^2 for i in 1:length(x)-1)
end

bounds_rosen = [(-2.048, 2.048) for _ in 1:5]
r_rosen = bench("Rosenbrock", rosenbrock, bounds_rosen; optimum=0.0, n_runs=DEFAULT_RUNS)
push!(ALL_RESULTS, r_rosen)
print_result(r_rosen)

── Rosenbrock ──
   melhor obtido  : 2.503766e-02
   média ± dp     : 2.503766e-02 ± 4.249e-18
   erro absoluto  : 2.504e-02
   erro rel (%)   : 2.5038%
   tempo médio(s) : 1.61
   nfev           : 125640


## 5. Problema 3: Rastrigin (multimodal)

$$f(x) = 10n + \sum_{i=1}^{n}\left[x_i^2 - 10\cos(2\pi x_i)\right],
\quad x^* = 0,\ f^* = 0$$

In [5]:
function rastrigin(x::Vector{Float64})::Float64
    n = length(x)
    10.0 * n + sum(xi^2 - 10.0 * cos(2π * xi) for xi in x)
end

bounds_rast = [(-5.12, 5.12) for _ in 1:10]
cfg_rast = GIVPConfig(max_iterations=80, alpha=0.12, vnd_iterations=50,
    ils_iterations=10, early_stop_threshold=60)
r_rast = bench("Rastrigin", rastrigin, bounds_rast;
    optimum=0.0, n_runs=DEFAULT_RUNS, cfg=cfg_rast)
push!(ALL_RESULTS, r_rast)
print_result(r_rast)

── Rastrigin ──
   melhor obtido  : 1.015409e+00
   média ± dp     : 1.015409e+00 ± 0.000e+00
   erro absoluto  : 1.015e+00
   erro rel (%)   : 101.5409%
   tempo médio(s) : 2.37
   nfev           : 436955


## 6. Problema 4: Ackley

$$f(x) = -20\exp\!\left(-0.2\sqrt{\tfrac{1}{n}\sum x_i^2}\right)
         -\exp\!\left(\tfrac{1}{n}\sum\cos(2\pi x_i)\right) + 20 + e,
\quad f^* = 0$$

In [6]:
function ackley(x::Vector{Float64})::Float64
    n = length(x)
    sum_sq = sum(xi^2 for xi in x)
    sum_cos = sum(cos(2π * xi) for xi in x)
    -20.0 * exp(-0.2 * sqrt(sum_sq / n)) - exp(sum_cos / n) + 20.0 + ℯ
end

bounds_ackley = [(-32.768, 32.768) for _ in 1:10]
r_ackley = bench("Ackley", ackley, bounds_ackley;
    optimum=0.0, n_runs=DEFAULT_RUNS)
push!(ALL_RESULTS, r_ackley)
print_result(r_ackley)

── Ackley ──
   melhor obtido  : 3.245689e-01
   média ± dp     : 3.245689e-01 ± 0.000e+00
   erro absoluto  : 3.246e-01
   erro rel (%)   : 32.4569%
   tempo médio(s) : 1.41
   nfev           : 107087


## 7. Problema 5: Griewank

$$f(x) = \frac{1}{4000}\sum x_i^2 - \prod\cos\!\left(\frac{x_i}{\sqrt{i}}\right) + 1,
\quad x \in [-600, 600]^{10},\ f^* = 0$$

In [7]:
function griewank(x::Vector{Float64})::Float64
    sum_sq = sum(xi^2 / 4000.0 for xi in x)
    prod_cos = prod(cos(x[i] / sqrt(Float64(i))) for i in eachindex(x))
    sum_sq - prod_cos + 1.0
end

bounds_griewank = [(-600.0, 600.0) for _ in 1:10]
r_griewank = bench("Griewank", griewank, bounds_griewank;
    optimum=0.0, n_runs=DEFAULT_RUNS)
push!(ALL_RESULTS, r_griewank)
print_result(r_griewank)

── Griewank ──
   melhor obtido  : 4.090184e-01
   média ± dp     : 4.090184e-01 ± 6.799e-17
   erro absoluto  : 4.090e-01
   erro rel (%)   : 40.9018%
   tempo médio(s) : 1.38
   nfev           : 108871


## 8. Maximização com `direction=GIVPOptimizer.maximize`

Demonstra o uso do modo de maximização diretamente — sem precisar negar a função.

In [8]:
# Maximizar: f(x) = -(sum xi^2)  =>  ótimo em x=0, valor=0
neg_sphere(x::Vector{Float64}) = -sum(xi^2 for xi in x)

bounds_max = [(-5.12, 5.12) for _ in 1:5]
res_max = givp(neg_sphere, bounds_max;
    direction=GIVPOptimizer.maximize,
    config=GIVPConfig(max_iterations=60),
    seed=42)

@printf("best_fun  = %.6e  (valor de neg_sphere, maximizado)\n", res_max.fun)
@printf("nfev      = %d\n", res_max.nfev)
@printf("success   = %s\n", res_max.success)
@printf("-best_fun = %.6e  (equivale ao Sphere minimizado)\n", -res_max.fun)

best_fun  = -8.770947e-05  (valor de neg_sphere, maximizado)
nfev      = 242585
success   = true
-best_fun = 8.770947e-05  (equivale ao Sphere minimizado)


## 9. Problema Combinatório: Knapsack 0/1

Variáveis contínuas em $[0,1]$ — itens selecionados pelo limiar $> 0.5$.
Capacidade excedida é penalizada.

In [9]:
const KS_N = 15
const KS_CAPACITY = 100.0
const KS_VALUES = [55.0, 45.0, 90.0, 70.0, 40.0, 30.0, 80.0, 60.0, 50.0, 35.0, 20.0, 75.0, 65.0, 25.0, 85.0]
const KS_WEIGHTS = [20.0, 15.0, 40.0, 30.0, 10.0, 5.0, 35.0, 25.0, 18.0, 12.0, 8.0, 28.0, 22.0, 6.0, 38.0]

function knapsack_penalty(x::Vector{Float64})::Float64
    selected = x .> 0.5
    total_value = sum(KS_VALUES[i] for i in 1:KS_N if selected[i]; init=0.0)
    total_weight = sum(KS_WEIGHTS[i] for i in 1:KS_N if selected[i]; init=0.0)
    overflow = max(0.0, total_weight - KS_CAPACITY)
    -total_value + 1000.0 * overflow
end

bounds_ks = [(0.0, 1.0) for _ in 1:KS_N]
r_ks = bench("Knapsack 0/1", knapsack_penalty, bounds_ks;
    optimum=-500.0,  # estimativa; DP daria o exato
    n_runs=DEFAULT_RUNS, rel_tol=500.0)
push!(ALL_RESULTS, r_ks)
print_result(r_ks)

selected_items = findall(r_ks.best_x .> 0.5)
@printf("   itens selecionados : %s\n", string(selected_items))
@printf("   valor total        : %.0f\n", sum(KS_VALUES[i] for i in selected_items; init=0.0))
@printf("   peso total         : %.0f  (cap=%.0f)\n",
    sum(KS_WEIGHTS[i] for i in selected_items; init=0.0), KS_CAPACITY)

── Knapsack 0/1 ──
   melhor obtido  : -3.150000e+02
   média ± dp     : -3.150000e+02 ± 0.000e+00
   erro absoluto  : 1.850e+02
   erro rel (%)   : 37.0000%
   tempo médio(s) : 1.13
   nfev           : 86805
   itens selecionados : [1, 2, 5, 6, 10, 11, 13, 14]
   valor total        : 315
   peso total         : 98  (cap=100)


## 10. Tabela Consolidada de Resultados

In [10]:
println("\n", repeat("─", 78))
@printf("%-20s  %12s  %12s  %10s  %10s  %8s\n",
        "Problema", "Melhor", "Média", "DP", "Erro Rel(%)", "Tempo(s)")
println(repeat("─", 78))
for r in ALL_RESULTS
        @printf("%-20s  %12.4e  %12.4e  %10.3e  %10.4f%%  %8.2f\n",
                r.problem, r.best_fun, r.mean_fun, r.std_fun, r.err_rel_pct, r.mean_time_s)
end
println(repeat("─", 78))


──────────────────────────────────────────────────────────────────────────────
Problema                    Melhor         Média          DP  Erro Rel(%)  Tempo(s)
──────────────────────────────────────────────────────────────────────────────
Sphere                  2.0090e-03    2.0090e-03   0.000e+00      0.2009%      2.42
Rosenbrock              2.5038e-02    2.5038e-02   4.249e-18      2.5038%      1.61
Rastrigin               1.0154e+00    1.0154e+00   0.000e+00    101.5409%      2.37
Ackley                  3.2457e-01    3.2457e-01   0.000e+00     32.4569%      1.41
Griewank                4.0902e-01    4.0902e-01   6.799e-17     40.9018%      1.38
Knapsack 0/1           -3.1500e+02   -3.1500e+02   0.000e+00     37.0000%      1.13
──────────────────────────────────────────────────────────────────────────────
